In [1]:
pip install openai pydub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil
from pydub import AudioSegment

# Explicitly ensure PATH contains Homebrew and Conda bin directories
extra_paths = [
    "/opt/homebrew/bin",
    "/usr/local/bin",
    "/opt/miniconda3/envs/bootcamp-env/bin"
]
os.environ["PATH"] += ":" + ":".join(extra_paths)

# Directly assign the executables to pydub
ffmpeg_path = shutil.which("ffmpeg")
ffprobe_path = shutil.which("ffprobe")

if ffmpeg_path and ffprobe_path:
    AudioSegment.converter = ffmpeg_path
    AudioSegment.ffprobe = ffprobe_path
    print(f"FFmpeg located at: {ffmpeg_path}")
    print(f"FFprobe located at: {ffprobe_path}")
else:
    print("Warning: ffmpeg or ffprobe still not found in search paths.")

# Load and inspect audio file
audio_path = "audio/audioforw1d5lab.mp3"

if os.path.exists(audio_path):
    print(f"\nSuccess: Audio file found at {audio_path}")
    audio = AudioSegment.from_file(audio_path)
    print("--- Audio Properties ---")
    print(f"Duration: {len(audio) / 1000.0} seconds")
    print(f"Channels: {audio.channels} (1 = Mono, 2 = Stereo)")
    print(f"Sample Width: {audio.sample_width} bytes")
    print(f"Frame Rate: {audio.frame_rate} Hz")
else:
    print(f"Error: Could not find audio file at {audio_path}.")

FFmpeg located at: /opt/homebrew/bin/ffmpeg
FFprobe located at: /opt/homebrew/bin/ffprobe

Success: Audio file found at audio/audioforw1d5lab.mp3
--- Audio Properties ---
Duration: 571.698 seconds
Channels: 2 (1 = Mono, 2 = Stereo)
Sample Width: 2 bytes
Frame Rate: 24000 Hz


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydub import AudioSegment

# 1. Load API key from .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is missing. Check your .env file!")

client = OpenAI(api_key=api_key)

# 2. Extract the first 30 seconds to test without waiting for a full file
source_audio_path = "audio/audioforw1d5lab.mp3"
short_sample_path = "audio/short_sample.mp3"

audio = AudioSegment.from_file(source_audio_path)
short_audio = audio[:90000]  # First 90 seconds (90,000 milliseconds)
short_audio.export(short_sample_path, format="mp3")

print(f"Created short audio sample: {short_sample_path} ({len(short_audio)/1000:.1f}s)")

# 3. Request basic transcription from OpenAI Whisper API
with open(short_sample_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

# 4. Display the transcription
print("\n--- Basic Transcription Output ---")
print(transcription.text)

Created short audio sample: audio/short_sample.mp3 (90.0s)

--- Basic Transcription Output ---
THE FAMOUS SPEECH OF PATRICK HENRY TO THE VIRGINIA HOUSE OF BURGESSES GIVEN ON MARCH 23, 1775 AND ENTITLED GIVE ME LIBERTY OR GIVE ME DEATH. No man thinks more highly than I do of the patriotism, as well as abilities, of the very worthy gentlemen who have just addressed the House. But different men often see the same subject in different lights, and therefore I hope it will not be thought disrespectful to those gentlemen if, entertaining as I do, opinions of a character very opposite to theirs, I shall speak forth my sentiments freely and without reserve. This is no time for ceremony. The question before the House is one of awful moment to this country. For my own part I consider it as nothing less than a question of freedom or slavery, and in proportion to the magnitude of the subject, ought to be the freedom of the debate.


In [7]:
# --- STEP 4 & 5: PROMPTED VS UNPROMPTED TRANSCRIPTION ---

# Define the sample file we created in Step 3
short_sample_path = "audio/short_sample.mp3"

# 1. Unguided Transcription (no prompt)
with open(short_sample_path, "rb") as audio_file:
    unprompted_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )
unprompted_text = unprompted_response.text

# 2. Guided Transcription (with prompt)
# Provide context, domain terms, or proper nouns likely present in your audio
guide_prompt = "Patrick Henry, Virginia Convention, American Revolution, Liberty or Death, St. John's Church, Richmond."

with open(short_sample_path, "rb") as audio_file:
    prompted_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=guide_prompt
    )
prompted_text = prompted_response.text

# 3. Compare Results
print("=" * 60)
print("UNPROMPTED (UNGUIDED) TRANSCRIPTION:")
print("-" * 60)
print(unprompted_text)
print("\n" + "=" * 60)
print("PROMPTED (GUIDED) TRANSCRIPTION:")
print("-" * 60)
print(prompted_text)
print("=" * 60)

# Optional quick diff check
if unprompted_text == prompted_text:
    print("\nObservation: Both outputs are identical.")
else:
    print("\nObservation: Differences detected between unguided and guided transcriptions.")

UNPROMPTED (UNGUIDED) TRANSCRIPTION:
------------------------------------------------------------
THE FAMOUS SPEECH OF PATRICK HENRY TO THE VIRGINIA HOUSE OF BURGESSES GIVEN ON MARCH 23, 1775 AND ENTITLED GIVE ME LIBERTY OR GIVE ME DEATH. No man thinks more highly than I do of the patriotism, as well as abilities, of the very worthy gentlemen who have just addressed the House. But different men often see the same subject in different lights, and therefore I hope it will not be thought disrespectful to those gentlemen if, entertaining as I do, opinions of a character very opposite to theirs, I shall speak forth my sentiments freely and without reserve. This is no time for ceremony. The question before the House is one of awful moment to this country. For my own part I consider it as nothing less than a question of freedom or slavery, and in proportion to the magnitude of the subject, ought to be the freedom of the debate.

PROMPTED (GUIDED) TRANSCRIPTION:
-------------------------------

In [8]:
import os

os.makedirs("chunks", exist_ok=True)

In [9]:
import os
import math
from pydub import AudioSegment
from openai import OpenAI

client = OpenAI()

def split_and_transcribe_audio(file_path, chunk_length_ms=60000):
    """
    Splits audio into segments of chunk_length_ms (default: 60 seconds),
    transcribes each chunk with verbose_json, and offsets timestamps.
    """
    audio = AudioSegment.from_file(file_path)
    total_duration_ms = len(audio)
    num_chunks = math.ceil(total_duration_ms / chunk_length_ms)
    
    print(f"Total audio duration: {total_duration_ms / 1000.0:.2f}s")
    print(f"Splitting into {num_chunks} chunks of {chunk_length_ms / 1000.0}s each...\n")

    combined_segments = []
    full_transcript_text = []

    for i in range(num_chunks):
        start_ms = i * chunk_length_ms
        end_ms = min((i + 1) * chunk_length_ms, total_duration_ms)
        chunk_offset_sec = start_ms / 1000.0

        # Export individual chunk
        chunk = audio[start_ms:end_ms]
        chunk_path = f"chunks/chunk_{i}.mp3"
        chunk.export(chunk_path, format="mp3")

        print(f"Transcribing chunk {i + 1}/{num_chunks} [{chunk_offset_sec:.1f}s - {end_ms/1000.0:.1f}s]...")

        # Request verbose_json to obtain timestamp segments
        with open(chunk_path, "rb") as audio_file:
            response = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                response_format="verbose_json",
                timestamp_granularities=["segment"]
            )

        full_transcript_text.append(response.text)

        # Offset the timestamps by chunk start position
        if hasattr(response, "segments") and response.segments:
            for seg in response.segments:
                seg_dict = seg if isinstance(seg, dict) else seg.model_dump()
                combined_segments.append({
                    "start": round(seg_dict["start"] + chunk_offset_sec, 2),
                    "end": round(seg_dict["end"] + chunk_offset_sec, 2),
                    "text": seg_dict["text"].strip()
                })

    return {
        "text": " ".join(full_transcript_text),
        "segments": combined_segments
    }

# Run the chunked transcription on your full audio file
result = split_and_transcribe_audio("audio/audioforw1d5lab.mp3", chunk_length_ms=60000)

print("\n--- Chunked Transcription Complete ---")
print(f"Total Segments Extracted: {len(result['segments'])}")
print("\nSample Timestamps (First 5 segments):")
for s in result["segments"][:5]:
    print(f"[{s['start']:06.2f}s -> {s['end']:06.2f}s] {s['text']}")

Total audio duration: 571.70s
Splitting into 10 chunks of 60.0s each...

Transcribing chunk 1/10 [0.0s - 60.0s]...
Transcribing chunk 2/10 [60.0s - 120.0s]...
Transcribing chunk 3/10 [120.0s - 180.0s]...
Transcribing chunk 4/10 [180.0s - 240.0s]...
Transcribing chunk 5/10 [240.0s - 300.0s]...
Transcribing chunk 6/10 [300.0s - 360.0s]...
Transcribing chunk 7/10 [360.0s - 420.0s]...
Transcribing chunk 8/10 [420.0s - 480.0s]...
Transcribing chunk 9/10 [480.0s - 540.0s]...
Transcribing chunk 10/10 [540.0s - 571.7s]...

--- Chunked Transcription Complete ---
Total Segments Extracted: 112

Sample Timestamps (First 5 segments):
[000.00s -> 024.72s] THE FAMOUS SPEECH OF PATRICK HENRY TO THE VIRGINIA HOUSE OF BURGESSES
[024.72s -> 038.62s] GIVEN ON MARCH 23, 1775 AND ENTITLED GIVE ME LIBERTY OR GIVE ME DEATH.
[038.62s -> 044.60s] No man thinks more highly than I do of the patriotism, as well as abilities, of the very
[044.60s -> 050.34s] worthy gentlemen who have just addressed the House. But d

In [10]:
import os
os.makedirs("exports", exist_ok=True)

In [11]:
import json

def format_timestamp_srt(seconds):
    """Converts seconds (float) into SRT format: HH:MM:SS,mmm"""
    millis = int(round((seconds - int(seconds)) * 1000))
    total_seconds = int(seconds)
    secs = total_seconds % 60
    mins = (total_seconds // 60) % 60
    hours = total_seconds // 3600
    return f"{hours:02d}:{mins:02d}:{secs:02d},{millis:03d}"

def format_timestamp_text(seconds):
    """Converts seconds into simple [HH:MM:SS] format"""
    total_seconds = int(seconds)
    secs = total_seconds % 60
    mins = (total_seconds // 60) % 60
    hours = total_seconds // 3600
    return f"[{hours:02d}:{mins:02d}:{secs:02d}]"

# 1. Export to JSON
json_output_path = "exports/transcription.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)
print(f"Exported JSON: {json_output_path}")

# 2. Export to Human-Readable TXT
txt_output_path = "exports/transcription.txt"
with open(txt_output_path, "w", encoding="utf-8") as f:
    f.write("--- AUDIO TRANSCRIPTION WITH TIMESTAMPS ---\n\n")
    for seg in result["segments"]:
        start_str = format_timestamp_text(seg["start"])
        end_str = format_timestamp_text(seg["end"])
        f.write(f"{start_str} - {end_str} {seg['text']}\n")
print(f"Exported TXT:  {txt_output_path}")

# 3. Export to SRT Subtitles
srt_output_path = "exports/transcription.srt"
with open(srt_output_path, "w", encoding="utf-8") as f:
    for idx, seg in enumerate(result["segments"], start=1):
        start_srt = format_timestamp_srt(seg["start"])
        end_srt = format_timestamp_srt(seg["end"])
        f.write(f"{idx}\n")
        f.write(f"{start_srt} --> {end_srt}\n")
        f.write(f"{seg['text']}\n\n")
print(f"Exported SRT:  {srt_output_path}")

Exported JSON: exports/transcription.json
Exported TXT:  exports/transcription.txt
Exported SRT:  exports/transcription.srt
